# Fine-tuning sBERT - Comparación de Accuracy

**Objetivo**: Implementar fine-tuning del modelo sBERT y comparar accuracy con el modelo base

**Estrategia**: Fine-tuning supervisado con pares de ejemplos similares/diferentes

---

## 1. Importaciones y Configuración

In [19]:
import numpy as np
import pandas as pd
from pathlib import Path
import json
import pickle
import random
from sentence_transformers import SentenceTransformer, InputExample, losses
from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator
from torch.utils.data import DataLoader
from sklearn.preprocessing import LabelEncoder
from sklearn.neural_network import MLPClassifier
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import warnings
import torch

warnings.filterwarnings('ignore')
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

print("Librerías importadas correctamente")

Librerías importadas correctamente


## 2. Carga de Datos y Funciones Auxiliares

In [20]:
# Configuración de paths
DATA_DIR = Path("../data")
PP_DIR = DATA_DIR / "preprocessed"
TRAIN_PP = PP_DIR / "train_preprocessed.csv"
TEST_PP = PP_DIR / "test_preprocessed.csv"
LABEL_MAPS = PP_DIR / "label_maps.json"

OUTPUT_DIR = Path("./model_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Cargar datos
train = pd.read_csv(TRAIN_PP)
test = pd.read_csv(TEST_PP)

with open(LABEL_MAPS, "r", encoding="utf-8") as f:
    label_maps = json.load(f)

print(f"Datos cargados - Train: {train.shape}, Test: {test.shape}")

Datos cargados - Train: (36696, 15), Test: (3, 11)


In [21]:
# Función para crear texto de entrada
def create_input_text(row):
    """Combina Question + Answer + Explanation para sBERT"""
    parts = []
    if pd.notna(row.get('QuestionText')):
        parts.append(str(row['QuestionText']))
    if pd.notna(row.get('MC_Answer')):
        parts.append(f"Answer: {row['MC_Answer']}")
    if pd.notna(row.get('StudentExplanation_clean')):
        parts.append(f"Explanation: {row['StudentExplanation_clean']}")
    return " ".join(parts)

# Crear textos de entrada
train['input_text'] = train.apply(create_input_text, axis=1)
test['input_text'] = test.apply(create_input_text, axis=1)

print(f"Texto de entrada creado")
print(f"Ejemplo: {train['input_text'].iloc[0][:150]}...")

Texto de entrada creado
Ejemplo: What fraction of the shape is not shaded? Give your answer in its simplest form. [Image: A triangle split into 9 equal smaller triangles. 6 of them ar...


## 3. Baseline - Modelo Sin Fine-tuning

In [22]:
print("=== BASELINE: Modelo Sin Fine-tuning ===")

# Cargar modelo pre-entrenado
model_name = 'all-MiniLM-L6-v2'
base_model = SentenceTransformer(model_name)

print(f"Modelo base cargado: {model_name}")
print(f"Dimensión embeddings: {base_model.get_sentence_embedding_dimension()}")

=== BASELINE: Modelo Sin Fine-tuning ===
Modelo base cargado: all-MiniLM-L6-v2
Dimensión embeddings: 384


In [23]:
# Generar embeddings con modelo base
print("\nGenerando embeddings con modelo base...")
X_train_base = base_model.encode(
    train['input_text'].tolist()[:5000],  # Usar muestra para velocidad
    show_progress_bar=True,
    batch_size=16,
    convert_to_numpy=True
)

# Preparar labels
label_encoder_category = LabelEncoder()
y_train_category_base = label_encoder_category.fit_transform(train['Category'][:5000])

print(f"Embeddings base generados: {X_train_base.shape}")
print(f"Categories: {len(label_encoder_category.classes_)} clases")


Generando embeddings con modelo base...


Batches:   0%|          | 0/313 [00:00<?, ?it/s]

Batches: 100%|██████████| 313/313 [00:05<00:00, 57.11it/s]


Embeddings base generados: (5000, 384)
Categories: 6 clases


In [24]:
# División train/validation para baseline
X_train_base_split, X_val_base, y_train_base_split, y_val_base = train_test_split(
    X_train_base, y_train_category_base,
    test_size=0.2, random_state=42, stratify=y_train_category_base
)

print(f"Train baseline: {X_train_base_split.shape[0]} muestras")
print(f"Validation baseline: {X_val_base.shape[0]} muestras")

Train baseline: 4000 muestras
Validation baseline: 1000 muestras


In [25]:
# Entrenar clasificador MLP con modelo base
print("\nEntrenando clasificador MLP con embeddings base...")
baseline_classifier = MLPClassifier(
    hidden_layer_sizes=(128, 64), 
    activation='relu', 
    solver='adam',
    alpha=0.0001, 
    batch_size=128, 
    learning_rate='adaptive', 
    max_iter=30,
    random_state=42, 
    early_stopping=True, 
    validation_fraction=0.1, 
    n_iter_no_change=5
)

baseline_classifier.fit(X_train_base_split, y_train_base_split)
y_pred_baseline = baseline_classifier.predict(X_val_base)
baseline_accuracy = accuracy_score(y_val_base, y_pred_baseline)

print(f"\n BASELINE ACCURACY: {baseline_accuracy:.4f} ({baseline_accuracy*100:.2f}%)")
print(f"Clases predichas: {len(set(y_pred_baseline))}")


Entrenando clasificador MLP con embeddings base...

 BASELINE ACCURACY: 0.7820 (78.20%)
Clases predichas: 4


## 4. Preparación de Datos para Fine-tuning

In [26]:
def create_training_pairs(df, max_pairs=5000):
    """
    Crear pares de entrenamiento para fine-tuning:
    - Pares positivos: misma Category (score = 1.0)
    - Pares negativos: diferente Category (score = 0.0)
    """
    training_examples = []
    
    # Agrupar por Category
    categories = df['Category'].unique()
    category_groups = {cat: df[df['Category'] == cat] for cat in categories}
    
    pairs_per_category = max_pairs // (len(categories) * 2)  # Dividir entre pos/neg
    
    for category in categories:
        group = category_groups[category]
        
        # Pares positivos (misma categoría)
        if len(group) >= 2:
            for _ in range(min(pairs_per_category, len(group) // 2)):
                idx1, idx2 = random.sample(list(group.index), 2)
                training_examples.append(InputExample(
                    texts=[df.loc[idx1, 'input_text'], df.loc[idx2, 'input_text']],
                    label=1.0
                ))
        
        # Pares negativos (diferentes categorías)
        other_categories = [c for c in categories if c != category]
        for _ in range(pairs_per_category):
            other_cat = random.choice(other_categories)
            other_group = category_groups[other_cat]
            
            if len(group) > 0 and len(other_group) > 0:
                idx1 = random.choice(list(group.index))
                idx2 = random.choice(list(other_group.index))
                training_examples.append(InputExample(
                    texts=[df.loc[idx1, 'input_text'], df.loc[idx2, 'input_text']],
                    label=0.0
                ))
    
    return training_examples

# Crear pares de entrenamiento
print("Creando pares de entrenamiento para fine-tuning...")
train_subset = train[:5000]  # Usar subset para velocidad
training_pairs = create_training_pairs(train_subset, max_pairs=3000)

print(f"Pares de entrenamiento creados: {len(training_pairs)}")
print(f"Ejemplo positivo: Score = {training_pairs[0].label}")
print(f"Ejemplo negativo: Score = {training_pairs[-1].label}")

Creando pares de entrenamiento para fine-tuning...
Pares de entrenamiento creados: 2526
Ejemplo positivo: Score = 1.0
Ejemplo negativo: Score = 0.0


## 5. Fine-tuning del Modelo sBERT

In [27]:
print("=== FINE-TUNING del Modelo sBERT ===")

# Crear modelo para fine-tuning
finetuned_model = SentenceTransformer(model_name)

# Crear DataLoader
train_dataloader = DataLoader(training_pairs, shuffle=True, batch_size=16)

# Configurar loss function
train_loss = losses.CosineSimilarityLoss(finetuned_model)

print(f"DataLoader creado: {len(train_dataloader)} batches")
print(f"Loss function: CosineSimilarityLoss")

=== FINE-TUNING del Modelo sBERT ===
DataLoader creado: 158 batches
Loss function: CosineSimilarityLoss


In [28]:
# Crear evaluador
val_pairs = training_pairs[:200]  # Usar subset para evaluación
val_sentences1 = [example.texts[0] for example in val_pairs]
val_sentences2 = [example.texts[1] for example in val_pairs]
val_scores = [example.label for example in val_pairs]

evaluator = EmbeddingSimilarityEvaluator(
    val_sentences1, val_sentences2, val_scores,
    name='validation'
)

print(f"Evaluador creado con {len(val_pairs)} pares de validación")

Evaluador creado con 200 pares de validación


In [29]:
# Fine-tuning
print("\nIniciando fine-tuning...")
finetuned_model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    evaluator=evaluator,
    epochs=3,
    evaluation_steps=100,
    warmup_steps=100,
    output_path=str(OUTPUT_DIR / "finetuned_sbert"),
    save_best_model=True,
    show_progress_bar=True
)

print("\n Fine-tuning completado!")
print(f"Modelo guardado en: {OUTPUT_DIR / 'finetuned_sbert'}")


Iniciando fine-tuning...


Step,Training Loss,Validation Loss,Validation Pearson Cosine,Validation Spearman Cosine
100,No log,No log,nan,nan
158,No log,No log,nan,nan
200,No log,No log,nan,nan
300,No log,No log,nan,nan
316,No log,No log,nan,nan
400,No log,No log,nan,nan
474,No log,No log,nan,nan



 Fine-tuning completado!
Modelo guardado en: model_outputs/finetuned_sbert


## 6. Evaluación del Modelo Fine-tuned

In [30]:
print("=== EVALUACIÓN: Modelo Fine-tuned ===")

# Cargar el mejor modelo fine-tuned
best_model = SentenceTransformer(str(OUTPUT_DIR / "finetuned_sbert"))

print(f"Modelo fine-tuned cargado desde: {OUTPUT_DIR / 'finetuned_sbert'}")

=== EVALUACIÓN: Modelo Fine-tuned ===
Modelo fine-tuned cargado desde: model_outputs/finetuned_sbert


In [31]:
# Generar embeddings con modelo fine-tuned
print("\nGenerando embeddings con modelo fine-tuned...")
X_train_finetuned = best_model.encode(
    train['input_text'].tolist()[:5000],
    show_progress_bar=True,
    batch_size=16,
    convert_to_numpy=True
)

print(f"Embeddings fine-tuned generados: {X_train_finetuned.shape}")


Generando embeddings con modelo fine-tuned...


Batches: 100%|██████████| 313/313 [00:05<00:00, 56.78it/s]

Embeddings fine-tuned generados: (5000, 384)


In [32]:
# División train/validation para modelo fine-tuned
X_train_ft_split, X_val_ft, y_train_ft_split, y_val_ft = train_test_split(
    X_train_finetuned, y_train_category_base,
    test_size=0.2, random_state=42, stratify=y_train_category_base
)

print(f"Train fine-tuned: {X_train_ft_split.shape[0]} muestras")
print(f"Validation fine-tuned: {X_val_ft.shape[0]} muestras")

Train fine-tuned: 4000 muestras
Validation fine-tuned: 1000 muestras


In [33]:
# Entrenar clasificador MLP con embeddings fine-tuned
print("\nEntrenando clasificador MLP con embeddings fine-tuned...")
finetuned_classifier = MLPClassifier(
    hidden_layer_sizes=(128, 64), 
    activation='relu', 
    solver='adam',
    alpha=0.0001, 
    batch_size=128, 
    learning_rate='adaptive', 
    max_iter=30,
    random_state=42, 
    early_stopping=True, 
    validation_fraction=0.1, 
    n_iter_no_change=5
)

finetuned_classifier.fit(X_train_ft_split, y_train_ft_split)
y_pred_finetuned = finetuned_classifier.predict(X_val_ft)
finetuned_accuracy = accuracy_score(y_val_ft, y_pred_finetuned)

print(f"\n FINE-TUNED ACCURACY: {finetuned_accuracy:.4f} ({finetuned_accuracy*100:.2f}%)")
print(f"Clases predichas: {len(set(y_pred_finetuned))}")


Entrenando clasificador MLP con embeddings fine-tuned...

 FINE-TUNED ACCURACY: 0.8500 (85.00%)
Clases predichas: 5


## 7. Comparación de Resultados

In [34]:
print("\n" + "="*60)
print("📊 COMPARACIÓN FINAL DE ACCURACY")
print("="*60)

improvement = finetuned_accuracy - baseline_accuracy
improvement_percent = (improvement / baseline_accuracy) * 100

print(f"\n Modelo Base (sin fine-tuning):")
print(f"   Accuracy: {baseline_accuracy:.4f} ({baseline_accuracy*100:.2f}%)")

print(f"\n Modelo Fine-tuned:")
print(f"   Accuracy: {finetuned_accuracy:.4f} ({finetuned_accuracy*100:.2f}%)")

print(f"\n Mejora:")
print(f"   Absoluta: +{improvement:.4f}")
print(f"   Relativa: +{improvement_percent:.2f}%")

if improvement > 0:
    print(f"\n El fine-tuning MEJORA el modelo")
else:
    print(f"\n El fine-tuning NO mejora el modelo")

print("\n" + "="*60)


📊 COMPARACIÓN FINAL DE ACCURACY

 Modelo Base (sin fine-tuning):
   Accuracy: 0.7820 (78.20%)

 Modelo Fine-tuned:
   Accuracy: 0.8500 (85.00%)

 Mejora:
   Absoluta: +0.0680
   Relativa: +8.70%

 El fine-tuning MEJORA el modelo



## 8. Análisis Detallado

In [ ]:
# Reportes de clasificación
print("\nREPORTE DETALLADO - Modelo Base:")
print(classification_report(
    y_val_base, y_pred_baseline, 
    target_names=label_encoder_category.classes_,
    zero_division=0
))

print("\n REPORTE DETALLADO - Modelo Fine-tuned:")
print(classification_report(
    y_val_ft, y_pred_finetuned, 
    target_names=label_encoder_category.classes_,
    zero_division=0
))


📋 REPORTE DETALLADO - Modelo Base:
                     precision    recall  f1-score   support

      False_Correct       0.00      0.00      0.00         7
False_Misconception       0.78      0.92      0.84       343
      False_Neither       0.75      0.51      0.60       180
       True_Correct       0.82      0.91      0.87       344
 True_Misconception       0.00      0.00      0.00         3
       True_Neither       0.68      0.50      0.57       123

           accuracy                           0.78      1000
          macro avg       0.50      0.47      0.48      1000
       weighted avg       0.77      0.78      0.77      1000


📋 REPORTE DETALLADO - Modelo Fine-tuned:
                     precision    recall  f1-score   support

      False_Correct       0.57      0.57      0.57         7
False_Misconception       0.88      0.90      0.89       343
      False_Neither       0.80      0.77      0.78       180
       True_Correct       0.86      0.95      0.90       344
 Tr

## 9. Guardar Resultados y Modelos

In [36]:
# Guardar clasificadores
with open(OUTPUT_DIR / "baseline_classifier.pkl", "wb") as f:
    pickle.dump(baseline_classifier, f)

with open(OUTPUT_DIR / "finetuned_classifier.pkl", "wb") as f:
    pickle.dump(finetuned_classifier, f)

# Guardar resultados de comparación
results = {
    "baseline_accuracy": float(baseline_accuracy),
    "finetuned_accuracy": float(finetuned_accuracy),
    "improvement_absolute": float(improvement),
    "improvement_percent": float(improvement_percent),
    "model_name": model_name,
    "training_pairs": len(training_pairs),
    "epochs": 3
}

with open(OUTPUT_DIR / "comparison_results.json", "w") as f:
    json.dump(results, f, indent=2)

print(f"\n Modelos y resultados guardados en: {OUTPUT_DIR}")
print(f"   - baseline_classifier.pkl")
print(f"   - finetuned_classifier.pkl")
print(f"   - finetuned_sbert/ (modelo sBERT fine-tuned)")
print(f"   - comparison_results.json")


 Modelos y resultados guardados en: model_outputs
   - baseline_classifier.pkl
   - finetuned_classifier.pkl
   - finetuned_sbert/ (modelo sBERT fine-tuned)
   - comparison_results.json
